# Choosing a model for encrypted inference

The published system ran a Random Forest on the server, which forced the server to decrypt the
patient's data first. This notebook picks the model for the version that never decrypts on the
server, and checks that encrypted predictions match plaintext ones.

**The hard constraint.** Paillier encryption supports only two operations on ciphertexts: adding
two of them, and multiplying one by a plaintext constant. A model can run under Paillier only if its
output is a weighted sum of its inputs. A logistic regression score `w . x + b` is exactly that. A
Random Forest (comparisons and branching) and KNN (distances between encrypted points) are not.

**The question for this notebook.** Does restricting ourselves to a linear model cost accuracy?

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from train_encrypted_model import CATEGORICAL, build_pipeline, encode

df = pd.read_csv("data/heart-disease.csv")
X, y = df.drop("target", axis=1), df["target"]
numeric = [c for c in X.columns if c not in CATEGORICAL]
X.shape, y.value_counts().to_dict()

((303, 13), {1: 165, 0: 138})

## Evaluation protocol

Every model is scored with the same **repeated stratified 5-fold cross-validation** (10 repeats,
fixed seed): 50 fits per model, each tested on data it did not see. With only 303 records, a single
train/test split is too noisy to rank models, as the last section shows.

Candidates:

- **Random Forest (paper)**: the published model's hyperparameters (205 trees, otherwise sklearn
  defaults), seeded here so the figures are reproducible.
- **KNN**: the other baseline from the original notebook.
- **Logistic regression, 13 raw features**: the simplest Paillier-compatible model.
- **Logistic regression, one-hot**: categorical features (`cp`, `restecg`, `slope`, `ca`, `thal`)
  one-hot encoded. Their codes are labels, not quantities (`thal = 3` is not "three times"
  `thal = 1`), so a linear model should treat each value separately. The client does the encoding
  before encrypting, giving 27 encrypted inputs.

In [2]:
# The published model's hyperparameters (tag paper-v1): 205 trees, otherwise sklearn defaults.
rf_paper = RandomForestClassifier(n_estimators=205, random_state=0)

candidates = [
    ("Random Forest (paper)", clone(rf_paper), "no"),
    ("KNN (k=5, standardized)", make_pipeline(StandardScaler(), KNeighborsClassifier()), "no"),
    ("Logistic regression, 13 raw features", make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), "yes"),
    ("Logistic regression, one-hot", build_pipeline(numeric), "yes"),
]
metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

rows, raw = [], {}
for name, model, paillier_ok in candidates:
    scores = cross_validate(model, X, y, cv=cv, scoring=metrics)
    raw[name] = scores["test_accuracy"]
    rows.append({"model": name, "runs under Paillier": paillier_ok,
                 **{m: f"{scores[f'test_{m}'].mean():.3f} ± {scores[f'test_{m}'].std():.3f}" for m in metrics}})
comparison = pd.DataFrame(rows).set_index("model")
comparison

,runs under Paillier,accuracy,precision,recall,f1,roc_auc
model,,,,,,
Random Forest (paper),no,0.822 ± 0.046,0.826 ± 0.055,0.859 ± 0.065,0.840 ± 0.042,0.904 ± 0.034
"KNN (k=5, standardized)",no,0.816 ± 0.049,0.812 ± 0.055,0.869 ± 0.063,0.837 ± 0.042,0.877 ± 0.043
"Logistic regression, 13 raw features",yes,0.830 ± 0.040,0.816 ± 0.046,0.892 ± 0.063,0.850 ± 0.037,0.897 ± 0.039
"Logistic regression, one-hot",yes,0.851 ± 0.050,0.852 ± 0.065,0.886 ± 0.059,0.866 ± 0.044,0.918 ± 0.035


Is the gap between the one-hot logistic regression and the Random Forest real, or noise? Both were
scored on the same 50 folds, so we can compare them fold by fold.

In [3]:
diff = raw["Logistic regression, one-hot"] - raw["Random Forest (paper)"]
print(f"Mean accuracy gain over the Random Forest: {diff.mean():+.3f}")
print(f"Folds where logistic regression wins / ties / loses: "
      f"{(diff > 0).sum()} / {(diff == 0).sum()} / {(diff < 0).sum()} of {len(diff)}")

Mean accuracy gain over the Random Forest: +0.029
Folds where logistic regression wins / ties / loses: 33 / 10 / 7 of 50


## Choosing the regularization strength

`C` is the inverse of the L2 penalty. Scored with the same protocol:

In [4]:
pd.DataFrame(
    [{"C": C, "accuracy": cross_val_score(
        make_pipeline(build_pipeline(numeric).steps[0][1], LogisticRegression(C=C, max_iter=5000)),
        X, y, cv=cv).mean().round(4)} for C in [0.03, 0.1, 0.3, 1, 3]]
).set_index("C")

,accuracy
C,
0.03,0.8297
0.10,0.8446
0.30,0.8508
1.00,0.8462
3.00,0.8422


## Why the paper's 91.8% is not comparable

The paper's training notebooks (tag `paper-v1`) report 91.8% and 88.5%, each from a single 80/20 split (61 test records).
Here is how much a single split moves the Random Forest's accuracy, depending only on the random
seed:

In [5]:
single = []
for seed in range(30):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    single.append(clone(rf_paper).set_params(random_state=0).fit(X_tr, y_tr).score(X_te, y_te))
single = np.array(single)
print(f"Random Forest, 30 single splits: min {single.min():.3f}, median {np.median(single):.3f}, max {single.max():.3f}")

Random Forest, 30 single splits: min 0.721, median 0.803, max 0.902


## Encrypted inference matches plaintext

The deployed model is trained by `train_encrypted_model.py`, which folds the standardization into
the weights and writes them to `he_services/ml_models/`. Below, every one of the 303 records is
encrypted with a fresh 2048-bit key, sent through the real Flask endpoint (in-process test client),
and decrypted. The server receives only the public key and ciphertexts.

This takes a few minutes with `gmpy2` installed, and about an hour without it.

In [6]:
import sys, time
import phe as paillier

sys.path.insert(0, "../he_services")
from api.ml.classification import MyClassification
from app import app

client = app.test_client()
url = "/api/operations/ml/classification"
model = MyClassification.load_model()
features = client.get(url).get_json()["features"]
w, b = np.array(model["weights"]), model["intercept"]
public_key, private_key = paillier.generate_paillier_keypair(n_length=2048)

start = time.perf_counter()
errors, agree = [], 0
for _, row in X.iterrows():
    x = encode(row, features)
    body = {"public_key": {"n": str(public_key.n)},
            "values": [(str(c.ciphertext()), c.exponent) for c in map(public_key.encrypt, x.tolist())]}
    assert "private_key" not in body
    c, exponent = client.post(url, json=body).get_json()["values"][0]
    score = private_key.decrypt(paillier.EncryptedNumber(public_key, int(c), int(exponent)))
    expected = float(x @ w + b)
    errors.append(abs(score - expected))
    agree += (score > 0) == (expected > 0)

print(f"{agree}/{len(X)} encrypted predictions identical to plaintext")
print(f"max score error: {max(errors):.2e}")
print(f"elapsed: {time.perf_counter() - start:.0f}s")

303/303 encrypted predictions identical to plaintext
max score error: 1.78e-15
elapsed: 147s


## Conclusion

**Deploy the one-hot logistic regression.**

- It runs under Paillier, which the Random Forest and KNN cannot. This alone rules them out for a
  server that must never decrypt.
- Restricting to a linear model costs nothing: the one-hot logistic regression is the best of the
  four on accuracy (85.1%), precision, F1 and ROC AUC (0.918), and beats the paper's Random Forest
  in 33 of the 50 folds, losing in 7.
- One-hot encoding is worth its cost (27 encrypted inputs instead of 13): +2.1 points over the
  logistic regression on raw codes, because the categorical codes are labels, not quantities.
- Recall is about 0.89 for both logistic regressions versus 0.86 for the Random Forest. For a
  screening tool, missing fewer patients with heart disease is the error that matters most.
- `C = 0.3` is the best of the grid, though anything from 0.1 to 1 is within half a point.
- Encryption changes nothing: all 303 encrypted predictions match plaintext, with score errors at
  floating-point rounding level.

**Caveats.** The ± figures are the spread across folds, not confidence intervals, and folds
overlap between repeats. 303 records from one hospital say little about other populations.
`C` was chosen with the same folds it is reported on, which flatters it slightly; the differences
across the grid are small enough that this does not change the choice.